In [2]:
import numpy as np
import torch
import torch.nn as nn

## Integer Positional Embedding

In [3]:
class IntegerPositionalEmbedding(nn.Module):
    def __init__(self, max_seq_length, embedding_dim, scale=1.0):
        super().__init__()
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.scale = scale
        self.embeddings = nn.Embedding(max_seq_length, embedding_dim)
        
    def forward(self, x):
        seq_length = x.shape[1]
        positions = torch.arange(seq_length)
        pos_embed = self.embeddings(positions) * self.scale
        return x + pos_embed.unsqueeze(0)

batch_size, seq_length, embedding_dim = 2, 10, 64
x = torch.randn(batch_size, seq_length, embedding_dim)

integer_pe = IntegerPositionalEmbedding(max_seq_length=512, embedding_dim=embedding_dim)
output = integer_pe(x)

print("Integer Positional Embedding:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")


Integer Positional Embedding:
Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])


## Binary Positional Embedding

In [4]:
class BinaryPositionalEmbedding(nn.Module):
    def __init__(self, max_seq_length, embedding_dim):
        super().__init__()
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        
        self.register_buffer('binary_embeddings', self._compute_binary_embeddings(max_seq_length, embedding_dim))
    
    def _compute_binary_embeddings(self, max_seq_length, embedding_dim):
        binary_embed = torch.zeros(max_seq_length, embedding_dim)
        
        for pos in range(max_seq_length):
            binary_str = bin(pos)[2:].zfill(embedding_dim)
            for i, bit in enumerate(binary_str):
                binary_embed[pos, i] = float(bit)
        
        return binary_embed
    
    def forward(self, x):
        seq_length = x.shape[1]
        pos_embed = self.binary_embeddings[:seq_length, :x.shape[-1]]
        return x + pos_embed.unsqueeze(0)

binary_pe = BinaryPositionalEmbedding(max_seq_length=512, embedding_dim=embedding_dim)
output = binary_pe(x)

print("Binary Positional Embedding:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")


Binary Positional Embedding:
Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])


## Sinusoidal Positional Embedding


In [5]:
class SinusoidalPositionalEmbedding(nn.Module):
    """
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, max_seq_length, embedding_dim):
        super().__init__()
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        
        self.register_buffer('pe', self._compute_pe(max_seq_length, embedding_dim))
    
    def _compute_pe(self, max_seq_length, embedding_dim):
        pe = torch.zeros(max_seq_length, embedding_dim)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * 
                           -(np.log(10000.0) / embedding_dim))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        
        if embedding_dim % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        return pe
    
    def forward(self, x):
        seq_length = x.shape[1]
        pos_embed = self.pe[:seq_length, :x.shape[-1]]
        return x + pos_embed.unsqueeze(0)

sinusoidal_pe = SinusoidalPositionalEmbedding(max_seq_length=512, embedding_dim=embedding_dim)
output = sinusoidal_pe(x)

print("Sinusoidal Positional Embedding:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")


Sinusoidal Positional Embedding:
Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])


In [11]:
import torch
import torch.nn as nn

class RotaryPositionalEncoding(nn.Module):
    def __init__(self, d_head, max_seq_len=2048):
        super().__init__()
        assert d_head % 2 == 0, "d_head must be even"

        self.d_head = d_head
        self.max_seq_len = max_seq_len

        theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer("theta", theta)

        positions = torch.arange(max_seq_len).unsqueeze(1)
        freqs = positions * theta.unsqueeze(0)

        self.register_buffer("cos", torch.cos(freqs))
        self.register_buffer("sin", torch.sin(freqs))

    def forward(self, q, k):
        seq_len = q.shape[2]

        cos = self.cos[:seq_len].to(q.device)[None, None, :, :]
        sin = self.sin[:seq_len].to(q.device)[None, None, :, :]

        def rotate(x):
            x1 = x[..., ::2]
            x2 = x[..., 1::2]
            return torch.cat([-x2, x1], dim=-1)

        q_rot = (q * cos.repeat_interleave(2, dim=-1)) + (rotate(q) * sin.repeat_interleave(2, dim=-1))
        k_rot = (k * cos.repeat_interleave(2, dim=-1)) + (rotate(k) * sin.repeat_interleave(2, dim=-1))

        return q_rot, k_rot
    
batch_size = 2
num_heads = 4
seq_len = 8
d_head = 16  # must be even

q = torch.randn(batch_size, num_heads, seq_len, d_head)
k = torch.randn(batch_size, num_heads, seq_len, d_head)

rope = RotaryPositionalEncoding(d_head,max_seq_len=512)

q_rot, k_rot = rope(q, k)

print("Rotary Positional Encoding:")
print(f"Input Q shape: {q.shape}")
print(f"Input K shape: {k.shape}")
print(f"Output Q_rot shape: {q_rot.shape}")
print(f"Output K_rot shape: {k_rot.shape}")

Rotary Positional Encoding:
Input Q shape: torch.Size([2, 4, 8, 16])
Input K shape: torch.Size([2, 4, 8, 16])
Output Q_rot shape: torch.Size([2, 4, 8, 16])
Output K_rot shape: torch.Size([2, 4, 8, 16])
